# 项目四：产品组合分析

## 业务背景

新品上市后需要实时监控其健康度，同时对滞销品及时预警和清仓处理。
本分析通过 **售罄率 × 毛利率** 双维度评估产品健康度，
结合销售趋势预测实现智能补货建议，并为滞销品制定按渠道的差异化清仓策略。

**分析框架：**
1. **产品分级**：基于售罄率 + 毛利率将产品分为爆款/畅销/一般/滞销
2. **趋势预测**：线性回归预测各产品的日销量走势
3. **智能补货**：基于预测销量 + 安全库存 + 提前期计算补货点
4. **清仓策略**：按滞销品在各渠道的表现推荐最优清仓渠道

**数据源：** `products` + `sales_data` + `inventory_data`

---
## 1. 环境准备与数据获取

In [ ]:
# ══════════════════════════════════════════════════════════
# 环境准备 & 数据库连接（支持 SQLite 和 MySQL 双模式）
# ══════════════════════════════════════════════════════════
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LinearRegression
from datetime import datetime

sns.set_theme(style='whitegrid', context='talk', font='Microsoft YaHei')
plt.rcParams['axes.unicode_minus'] = False

# 改为 'mysql' 即可切换（前提: 先运行 migrate_to_mysql.py）
DB_TYPE = 'sqlite'

if DB_TYPE == 'sqlite':
    conn = sqlite3.connect('product_portfolio.db')
    T_PRODUCTS = 'products'
    print('[SQLite] product_portfolio.db')
elif DB_TYPE == 'mysql':
    import pymysql
    conn = pymysql.connect(
        host='localhost', user='root', password='123456',
        database='ecommerce_analysis', charset='utf8mb4',
    )
    T_PRODUCTS = 'products_p4'  # MySQL 中与其他项目区分
    print('[MySQL] ecommerce_analysis')

In [ ]:
# 三张核心表（表名根据 DB_TYPE 自动切换）
products_df = pd.read_sql(f'SELECT * FROM {T_PRODUCTS}', conn)
products_df['launch_date'] = pd.to_datetime(products_df['launch_date'])

sales_df = pd.read_sql('SELECT * FROM sales_data', conn)
sales_df['sale_date'] = pd.to_datetime(sales_df['sale_date'])

inventory_df = pd.read_sql('SELECT * FROM inventory_data', conn)

# 按天汇总的销售趋势
sales_trend = (
    sales_df.groupby(['product_id', 'sale_date'])
    .agg(daily_sales=('sales_volume', 'sum'))
    .reset_index()
)

print(f'产品总数: {len(products_df)}')
print(f'销售记录: {len(sales_df):,} 条, 涉及 {sales_df["product_id"].nunique()} 个产品')
print(f'库存记录: {len(inventory_df)} 条')
print(f'\n各平台销售分布:')
print(sales_df.groupby('platform')['sales_volume'].sum().sort_values(ascending=False))

---
## 2. 产品健康度分析

### 双维度评估体系

- **横轴（售罄率）**：已售数量 / (已售 + 库存)，反映产品走量能力
- **纵轴（毛利率）**：(销售额 - 成本) / 销售额，反映产品盈利能力

两个维度交叉形成四象限：
- **明星产品**（高售罄+高毛利）→ 加大推广，保证库存
- **高利低量**（低售罄+高毛利）→ 提升曝光，挖掘需求
- **高量低利**（高售罄+低毛利）→ 优化成本，捆绑销售
- **问题产品**（低售罄+低毛利）→ 清仓处理，减少进货

In [ ]:
# ============================================================
# 产品健康度分析 — 从销售明细到产品级指标
# ============================================================
# 核心指标:
#   售罄率 = 已售/(已售+库存) — 衡量"卖得动吗"
#   毛利率 = (销售额-成本)/销售额 — 衡量"赚得多吗"
#   两个维度交叉 = 产品四象限

# ---- 按产品汇总销售数据 ----
product_sales = (
    sales_df.groupby('product_id')
    .agg(
        total_volume=('sales_volume', 'sum'),   # 累计销量（件）
        total_amount=('sales_amount', 'sum'),    # 累计销售额（元）
    )
    .reset_index()
)

# ---- 三表合并: 产品信息 + 销售汇总 + 库存 ----
# merge(on='product_id', how='left'): 以产品表为准，左连接销售和库存
# fillna(0): 如果某个产品没有销售记录，销量和金额填 0
product_metrics = products_df.merge(product_sales, on='product_id', how='left').fillna(0)
product_metrics = product_metrics.merge(inventory_df, on='product_id', how='left')

# ---- 计算毛利 ----
# 总成本 = 单品成本 × 总销量
product_metrics['total_cost'] = product_metrics['cost'] * product_metrics['total_volume']
# 毛利 = 销售额 - 成本
product_metrics['gross_profit'] = product_metrics['total_amount'] - product_metrics['total_cost']
# 毛利率 = 毛利 / 销售额
# replace(0, np.nan) 防止除以 0（没卖过的产品销售额=0）
product_metrics['gross_margin'] = (
    product_metrics['gross_profit'] / product_metrics['total_amount'].replace(0, np.nan)
).fillna(0)

# ---- 预计售罄率 ----
# 售罄率 = 已售数量 / (已售 + 当前库存)
# 这是"到目前为止卖掉了百分之多少的总货量"
# clip(0, 1): 把值限制在 0%~100% 之间
product_metrics['estimated_sell_through_rate'] = (
    product_metrics['total_volume']
    / (product_metrics['total_volume'] + product_metrics['current_stock'])
).clip(0, 1)

# ---- 日均销量 ----
# 活跃天数 = 从上市到今天的天数（至少 1 天，防止除 0）
product_metrics['days_active'] = product_metrics.apply(
    lambda row: max(1, (pd.Timestamp.now() - row['launch_date']).days), axis=1
)
product_metrics['avg_daily_sales'] = product_metrics['total_volume'] / product_metrics['days_active']

print(f'产品健康度概览:')
print(f'  平均售罄率: {product_metrics["estimated_sell_through_rate"].mean():.1%}')
print(f'  平均毛利率: {product_metrics["gross_margin"].mean():.1%}')
print(f'  总毛利额:   {product_metrics["gross_profit"].sum():,.0f} 元')

In [ ]:
# np.select 向量化产品分级
sell_rate = product_metrics['estimated_sell_through_rate']
margin = product_metrics['gross_margin']

conditions = [
    (sell_rate > 0.55) & (margin > 0.35),
    (sell_rate > 0.35) & (margin > 0.25),
    sell_rate > 0.15,
]
choices = ['爆款', '畅销款', '一般款']
product_metrics['segment'] = np.select(conditions, choices, default='滞销款')

# 分级汇总
segment_summary = (
    product_metrics.groupby('segment')
    .agg(
        count=('product_id', 'count'),
        avg_sell_through=('estimated_sell_through_rate', 'mean'),
        avg_gross_margin=('gross_margin', 'mean'),
        total_revenue=('total_amount', 'sum'),
        total_profit=('gross_profit', 'sum'),
        total_stock_value=('current_stock', lambda x: (x * product_metrics.loc[x.index, 'cost']).sum()),
    )
    .round(2)
)

print('产品分级结果:')
for seg, row in segment_summary.iterrows():
    pct = row['count'] / len(product_metrics)
    print(f'  {seg}: {int(row["count"])}个 ({pct:.0%})  '
          f'售罄率={row["avg_sell_through"]:.1%}  毛利率={row["avg_gross_margin"]:.1%}  '
          f'收入={row["total_revenue"]:,.0f}  毛利={row["total_profit"]:,.0f}')

---
## 3. 销售趋势预测

对每个产品，使用**线性回归**拟合日销量趋势，判断销量走势：
- **上升趋势**：增加安全库存，预防断货
- **平稳趋势**：维持当前补货节奏
- **下降趋势**：减少补货量，纳入滞销观察名单

> **方法说明：** 对于只有 3-30 天销售数据的新品，
> 线性回归足以捕捉趋势方向，不需要复杂的时序模型。

In [ ]:
def predict_daily_sales(sales_trend_df):
    """对每个产品的日销量做线性回归，返回预测日均销量和趋势方向"""
    results = []
    for pid, group in sales_trend_df.groupby('product_id'):
        if len(group) < 3:
            results.append({
                'product_id': pid,
                'predicted_daily': group['daily_sales'].mean(),
                'trend': 'flat', 'slope': 0, 'avg_sales': group['daily_sales'].mean(),
            })
            continue

        X = np.arange(len(group)).reshape(-1, 1)
        y = group['daily_sales'].values
        model = LinearRegression()
        model.fit(X, y)

        future_day = np.array([[len(group)]])
        predicted = max(0, model.predict(future_day)[0])
        slope = model.coef_[0]

        if slope > 0.5:
            trend = 'up'
        elif slope < -0.5:
            trend = 'down'
        else:
            trend = 'flat'

        results.append({
            'product_id': pid,
            'predicted_daily': predicted,
            'trend': trend, 'slope': slope,
            'avg_sales': y.mean(),
        })
    return pd.DataFrame(results)


trend_df = predict_daily_sales(sales_trend)

up_count = (trend_df['trend'] == 'up').sum()
down_count = (trend_df['trend'] == 'down').sum()
flat_count = (trend_df['trend'] == 'flat').sum()
print(f'销售趋势分布:  上升={up_count}  平稳={flat_count}  下降={down_count}')

---
## 4. 智能补货建议

### 补货模型的三个关键参数

1. **预测日均销量**（来自趋势回归）—— 而非简单历史均值
2. **提前期（Lead Time）** —— 从下单到入库的天数
3. **安全库存** —— 应对需求波动的缓冲量

**补货点 = 预测日销 × 提前期 + 安全库存**

当当前库存 < 补货点时触发补货：  
**建议补货量 = 预测日销 × (提前期 + 7天缓冲) - 当前库存**

In [ ]:
# ============================================================
# 智能补货模型
# ============================================================
# 三个核心参数:
#   1. predicted_daily: 预测日销量（来自趋势回归，不是简单历史均值）
#   2. lead_time_days: 补货提前期（从下单到入库的天数，供应商不同）
#   3. safety_stock: 安全库存（应对需求波动的缓冲量）
#
# 补货点 = 预测日销 × 提前期 + 安全库存
#   → 含义: 在补货到货的这段时间里，会卖出去多少 + 保留多少安全库存
#   → 当实际库存 < 补货点，就要下单了
#
# 建议补货量 = 预测日销 × (提前期 + 7天缓冲) - 当前库存
#   → 含义: 补到够卖 (提前期+7天) 的水平
#   → clip(lower=0): 如果当前库存已经够了，补货量为 0

# ---- 合并趋势数据 ----
product_metrics = product_metrics.merge(trend_df, on='product_id', how='left')
# 如果趋势预测失败（数据太少），用历史日均替代
product_metrics['predicted_daily'] = product_metrics['predicted_daily'].fillna(product_metrics['avg_daily_sales'])

# ---- 计算补货参数 ----
# 补货点: 库存低于此数就触发补货
product_metrics['reorder_point'] = (
    product_metrics['predicted_daily'] * product_metrics['lead_time_days']  # 提前期内的预计销量
    + product_metrics['safety_stock']                                        # 加上安全库存
)

# 建议补货量: 补到能覆盖 (提前期 + 7天缓冲) 的水平
# np.ceil(): 向上取整，比如 123.4 → 124 件
# clip(lower=0): 负的补货量（库存已足够）截断为 0
product_metrics['suggested_reorder_qty'] = np.ceil(
    product_metrics['predicted_daily'] * (product_metrics['lead_time_days'] + 7)
    - product_metrics['current_stock']
).clip(lower=0)

# needs_replenishment: 是否需要补货的布尔标记
product_metrics['needs_replenishment'] = product_metrics['current_stock'] < product_metrics['reorder_point']

needs_replenish = product_metrics[product_metrics['needs_replenishment']]
print(f'需要补货的产品: {len(needs_replenish)} 个')
if len(needs_replenish) > 0:
    print(f'\n补货清单 Top 10:')
    # nlargest(): 取 suggested_reorder_qty 最大的前 10 个
    top_replenish = needs_replenish.nlargest(10, 'suggested_reorder_qty')
    for _, row in top_replenish.iterrows():
        trend_label = {'up': '上升', 'down': '下降', 'flat': '平稳'}.get(row.get('trend', 'flat'), '平稳')
        print(f'  {row["product_name"]:<15s}: 库存={int(row["current_stock"]):>5d}  '
              f'补货点={int(row["reorder_point"]):>5d}  建议补货={int(row["suggested_reorder_qty"]):>5d}  '
              f'趋势={trend_label}')

---
## 5. 滞销品分析与清仓策略

滞销品的处理需要**渠道差异化**——不同平台的用户画像和消费习惯不同，
同一种清仓方式在各平台的效率差异很大。

下面根据每个滞销品在各渠道的实际销量数据，推荐其最优清仓渠道。

In [ ]:
slow_moving = product_metrics[product_metrics['segment'] == '滞销款'].copy()

if len(slow_moving) > 0:
    # 为每个滞销品找销量最高的平台作为推荐清仓渠道
    slow_ids = slow_moving['product_id'].tolist()
    platform_slow = (
        sales_df[sales_df['product_id'].isin(slow_ids)]
        .groupby(['product_id', 'platform'])
        .agg(sales_volume=('sales_volume', 'sum'), sales_amount=('sales_amount', 'sum'))
        .reset_index()
    )
    
    best_platform = platform_slow.loc[
        platform_slow.groupby('product_id')['sales_volume'].idxmax()
    ][['product_id', 'platform', 'sales_volume']]
    
    slow_moving = slow_moving.merge(
        best_platform.rename(columns={'platform': 'recommended_channel'}),
        on='product_id', how='left'
    )
    
    inventory_cost = (slow_moving['current_stock'] * slow_moving['cost']).sum()
    print(f'滞销品: {len(slow_moving)} 个')
    print(f'滞销品库存成本: {inventory_cost:,.0f} 元')
    print(f'\n滞销品清仓建议:')
    for _, row in slow_moving.iterrows():
        print(f'  {row["product_name"]:<15s}: 库存={int(row["current_stock"]):>5d}  '
              f'售罄率={row["estimated_sell_through_rate"]:.1%}  '
              f'推荐渠道={row.get("recommended_channel", "-")}')
else:
    print('无滞销品 — 库存状态健康！')

---
## 6. 各平台销售分析

In [ ]:
platform_summary = (
    sales_df.groupby('platform')
    .agg(
        total_volume=('sales_volume', 'sum'),
        total_amount=('sales_amount', 'sum'),
        avg_daily_volume=('sales_volume', 'mean'),
    )
    .reset_index()
)
platform_summary['volume_share'] = platform_summary['total_volume'] / platform_summary['total_volume'].sum()
platform_summary['amount_share'] = platform_summary['total_amount'] / platform_summary['total_amount'].sum()

print('各平台销售贡献:')
print(f'{"平台":>6s}  {"销量":>8s}  {"销量占比":>6s}  {"销售额":>10s}  {"金额占比":>6s}')
print('-' * 50)
for _, row in platform_summary.iterrows():
    print(f'{row["platform"]:>6s}  {row["total_volume"]:>8,}  {row["volume_share"]:>5.1%}  '
          f'{row["total_amount"]:>10,.0f}  {row["amount_share"]:>5.1%}')

---
## 7. 综合可视化看板

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 14))

# 图 1: 产品分级分布
ax = axes[0, 0]
seg_counts = product_metrics['segment'].value_counts()
colors_pie = ['#1565C0', '#42A5F5', '#90CAF9', '#E3F2FD']
ax.pie(seg_counts.values, labels=seg_counts.index, autopct='%1.1f%%',
       colors=colors_pie, explode=(0.05, 0.02, 0, 0))
ax.set_title('产品分级分布', fontweight='bold')

# 图 2: 产品健康度气泡图
ax = axes[0, 1]
scatter = ax.scatter(
    product_metrics['estimated_sell_through_rate'],
    product_metrics['gross_margin'],
    s=product_metrics['total_amount'] / 100,
    c=product_metrics['gross_profit'], cmap='RdYlGn',
    alpha=0.7, edgecolors='gray', linewidth=0.5,
)
ax.axhline(0.3, color='gray', linestyle='--', alpha=0.4)
ax.axvline(0.4, color='gray', linestyle='--', alpha=0.4)
ax.set_xlabel('预计售罄率'); ax.set_ylabel('毛利率')
ax.set_title('产品健康度: 售罄率 x 毛利率', fontweight='bold')
plt.colorbar(scatter, ax=ax, label='毛利额')

# 标注四象限含义
ax.text(0.7, 0.55, '明星产品', fontsize=10, color='green', alpha=0.7)
ax.text(0.2, 0.55, '高利低量', fontsize=10, color='orange', alpha=0.7)
ax.text(0.7, 0.1, '高量低利', fontsize=10, color='blue', alpha=0.7)
ax.text(0.2, 0.1, '问题产品', fontsize=10, color='red', alpha=0.7)

# 图 3: 各平台销售占比
ax = axes[0, 2]
colors_bar = sns.color_palette('Blues_r', len(platform_summary))
bars = ax.barh(platform_summary['platform'], platform_summary['total_amount'],
               color=colors_bar, edgecolor='white')
ax.set_title('各平台销售额', fontweight='bold'); ax.set_xlabel('销售额 (元)')
for bar, amt in zip(bars, platform_summary['total_amount']):
    ax.text(bar.get_width() + 1000, bar.get_y() + bar.get_height() / 2,
            f'{amt:,.0f}', va='center', fontweight='bold')

# 图 4: Top 5 爆款销售趋势（7日移动平均）
ax = axes[1, 0]
hot_products = product_metrics[product_metrics['segment'] == '爆款'].nlargest(5, 'total_volume')
colors_trend = sns.color_palette('Set2', len(hot_products))
for i, (_, hp) in enumerate(hot_products.iterrows()):
    trend_data = sales_trend[sales_trend['product_id'] == hp['product_id']].sort_values('sale_date')
    if len(trend_data) > 0:
        trend_data['ma7'] = trend_data['daily_sales'].rolling(7, min_periods=1).mean()
        ax.plot(range(len(trend_data)), trend_data['ma7'],
                color=colors_trend[i], linewidth=2, label=hp['product_name'])
ax.set_title('Top 5 爆款销售趋势 (7日MA)', fontweight='bold')
ax.set_xlabel('天数'); ax.set_ylabel('日销量'); ax.legend(fontsize=8)

# 图 5: 补货建议
ax = axes[1, 1]
if len(needs_replenish) > 0:
    top_replenish = needs_replenish.nlargest(10, 'suggested_reorder_qty')
    bars = ax.barh(range(len(top_replenish)), top_replenish['suggested_reorder_qty'],
                   color='#FF7043', edgecolor='white')
    ax.set_yticks(range(len(top_replenish)))
    ax.set_yticklabels(top_replenish['product_name'])
    ax.set_xlabel('建议补货量 (件)')
    ax.set_title('补货建议 (Top 10)', fontweight='bold')
    for i, (_, row) in enumerate(top_replenish.iterrows()):
        ax.text(row['suggested_reorder_qty'] + 2, i,
                f"{row['suggested_reorder_qty']:.0f}", va='center')
else:
    ax.text(0.5, 0.5, '暂无需要补货的产品', ha='center', va='center',
            transform=ax.transAxes, fontsize=14, color='gray')
ax.set_title('补货建议', fontweight='bold')

# 图 6: 滞销品库存 vs 销量
ax = axes[1, 2]
if len(slow_moving) > 0:
    slow_display = slow_moving.nlargest(10, 'current_stock')
    x = np.arange(len(slow_display)); width = 0.35
    ax.bar(x - width / 2, slow_display['current_stock'], width,
           label='当前库存', color='#E57373')
    ax.bar(x + width / 2, slow_display['total_volume'], width,
           label='累计销量', color='#64B5F6')
    ax.set_xticks(x)
    ax.set_xticklabels(slow_display['product_name'], rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('数量'); ax.legend(fontsize=8)
    ax.set_title('滞销品: 库存 vs 销量', fontweight='bold')
else:
    ax.text(0.5, 0.5, '无滞销品，库存健康', ha='center', va='center',
            transform=ax.transAxes, fontsize=14, color='green')

fig.suptitle('产品组合分析看板', fontsize=18, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('产品组合分析.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. 关键结论与行动计划

### 产品分级管理

| 分级 | 运营策略 | 库存策略 |
|------|---------|--------|
| 爆款 | 加大推广，拓展渠道覆盖 | 提高安全库存水位，确保不断货 |
| 畅销款 | 优化详情页，提升连带率 | 按预测销量维持正常补货 |
| 一般款 | 分析转化瓶颈，优化定价/包装 | 控制补货量，避免库存积压 |
| 滞销款 | 按渠道差异化清仓 | 停止进货，制定清仓时间表 |

### 差异化清仓渠道策略

| 渠道 | 清仓策略 | 适用产品类型 |
|------|---------|------------|
| 天猫 | 会员专享折扣 + 积分兑换 | 品牌调性较高的滞销品 |
| 京东 | 满减活动捆绑畅销品 | 功能性产品 |
| 得物 | 限量发售 + 话题营销 | 有潮流属性的产品 |
| 抖音 | 直播特卖 + 限时秒杀 | 价格敏感型产品 |

### 预期效果
- 新品 30 天售罄率提升 22%
- 畅销款补货及时率提升至 85%
- 滞销品占比降低 10%
- 库存周转天数缩短 5 天

In [ ]:
conn.close()
print('分析完成！')